2D Heat Diffusion on the GPU: all result graphs drawn from the Excel workbook heat_diffusion_graph_data.xlsx (CPU reference runs and Tesla T4 measurements).

In [ ]:
import os, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FuncFormatter, NullFormatter

XLSX = "heat_diffusion_graph_data.xlsx"
RAW = ("https://raw.githubusercontent.com/Manishrepo-bi/2D-Heat-Diffusion-on-GPU/main/results/" + XLSX)

if not os.path.exists(XLSX):
    try:
        urllib.request.urlretrieve(RAW, XLSX)
        print("Downloaded from GitHub")
    except Exception:
        try:
            from google.colab import files
            print("Choose heat_diffusion_graph_data.xlsx")
            up = files.upload()
            XLSX = next(iter(up))
        except ImportError:
            raise FileNotFoundError("Put heat_diffusion_graph_data.xlsx next to this notebook")

book = pd.read_excel(XLSX, sheet_name=None)
plates = {k: pd.read_excel(XLSX, sheet_name=f"Plate_{k}", header=None).to_numpy() for k in ("3a", "3b", "3c")}
for name, df in book.items():
    print(f"{name:20s} {df.shape[0]:>6} rows x {df.shape[1]} columns")

In [ ]:
SURFACE, INK, INK2, MUTED, GRID = "#ffffff", "#0b0b0b", "#52514e", "#8a8984", "#e6e5e0"
BACK = {"cpu_reference": "#2a78d6", "cuda_global": "#eb6834", "cuda_shared": "#1baf7a"}
BACK_LIGHT = {"cuda_global": "#f6b394", "cuda_shared": "#8fd9bd"}
BACK_NAME = {"cpu_reference": "CPU (NumPy)", "cuda_global": "GPU global memory", "cuda_shared": "GPU shared memory"}
CASE = {"3a uniform": "#6250d6", "3b edges": "#e87ba4", "3c hot edge": "#008300"}
EXACT = {"3a uniform": 75.0, "3b edges": 40.0, "3c hot edge": 25.0}
RAMP4 = ["#86b6ef", "#3987e5", "#1c5cab", "#0d366b"]
SEQ = LinearSegmentedColormap.from_list("blue_seq", ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"])
plt.rcParams.update({
    "font.size": 9, "axes.titlesize": 10, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.edgecolor": GRID, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.axisbelow": True, "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
    "legend.fontsize": 8, "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "lines.linewidth": 2,
    "lines.markersize": 5, "figure.dpi": 150,
})
fmt_int = FuncFormatter(lambda v, _=None: f"{int(v):,}" if v >= 1 else f"{v:g}")

def label(ax, x, y, text, dx=4, dy=0, ha="left"):
    ax.annotate(text, (x, y), xytext=(dx, dy), textcoords="offset points", color=INK, fontsize=8, va="center",
                ha=ha, bbox=dict(boxstyle="round,pad=0.15", fc=SURFACE, ec="none", alpha=0.85))

def show(fig):
    for ax in fig.axes:
        if ax.get_xscale() == "log":
            ax.xaxis.set_minor_formatter(NullFormatter())
    plt.show()

bench, tune = book["T4_benchmark"], book["T4_tuning"]
svn = book["Sweeps_vs_N"]
eps = book["Sweeps_vs_eps"]
NS = [9, 17, 33, 65, 129, 257]

def t4(backend, check=1, col="ms_per_sweep"):
    r = bench[(bench.backend == backend) & (bench.check_every == check)].sort_values("n")
    return r.n.to_numpy(), r[col].to_numpy()

Figure 1. Converged plates (sheets Plate_3a, Plate_3b, Plate_3c): 65 x 65 plates at eps = 1e-8. The dot marks the centre cell: exactly 75, 40 and 25.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.7), constrained_layout=True)
for ax, (k, case) in zip(axes, zip(("3a", "3b", "3c"), CASE)):
    g = plates[k]; c = g.shape[0] // 2
    im = ax.imshow(g, cmap=SEQ, vmin=0, vmax=100, interpolation="nearest")
    ax.plot(c, c, marker="o", ms=5, mfc=SURFACE, mec=INK, mew=1)
    ax.set_title(f"{case}: centre {g[c, c]:.2f}", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    for s in ax.spines.values(): s.set_visible(False)
cb = fig.colorbar(im, ax=axes, shrink=0.85, pad=0.02); cb.set_label("temperature", color=INK2); cb.outline.set_visible(False)
show(fig)

Figure 2. Temperature across the middle row (sheet Middle_row): 3a is flat at 75; in 3b heat flows from the hot top and left edges towards the cold right edge.

In [ ]:
mid = book["Middle_row"]
fig, ax = plt.subplots(figsize=(6.4, 2.9))
c = 32
for case in CASE:
    ax.plot(mid.j, mid[case], color=CASE[case])
    ax.plot(c, mid[case][c], "o", color=CASE[case], mec=SURFACE, mew=1.5, ms=6)
    label(ax, c, mid[case][c], f"{case}: centre {mid[case][c]:.0f}", dx=0, dy=10, ha="center")
ax.set_xlim(0, 64); ax.set_ylim(-3, 88)
ax.set_xlabel("column j along the middle row (65 x 65 grid)"); ax.set_ylabel("temperature")
ax.set_title("Temperature across the middle row")
show(fig)

Figure 3. Largest change per sweep (sheet Convergence_history): Each line falls at a steady rate on the log scale; larger grids fall far more slowly.

In [ ]:
h = book["Convergence_history"]
fig, ax = plt.subplots(figsize=(6.4, 3.1))
for col, name in zip(RAMP4, [c for c in h.columns if c != "sweep"]):
    s = h[name].dropna()
    ax.semilogy(h.sweep[: len(s)], s, color=col, lw=1.8, label=f"{name}: {len(s):,} sweeps")
ax.axhline(1e-6, color=MUTED, lw=1, ls="--"); ax.text(32000, 1.7e-6, "eps = 1e-6", ha="right", fontsize=8, color=INK2)
ax.set_xlim(0, 40000); ax.xaxis.set_major_formatter(fmt_int)
ax.legend(loc="upper right", title="grid size (to eps = 1e-6)", title_fontsize=8)
ax.set_xlabel("sweep"); ax.set_ylabel("largest change in the sweep")
ax.set_title("How fast the largest change shrinks (case 3c)")
show(fig)

Figure 4. Sweeps needed against grid size (sheet Sweeps_vs_N): Roughly 3x more sweeps for every doubling of N.

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 3.0))
for case, dy in zip(CASE, (7, 0, -7)):
    r = svn[svn.case == case].sort_values("n")
    ax.loglog(r.n, r.sweeps, "o-", color=CASE[case], mec=SURFACE, mew=1.2)
    label(ax, r.n.iloc[-1], r.sweeps.iloc[-1], case, dy=dy)
ax.set_xticks(NS); ax.xaxis.set_major_formatter(fmt_int); ax.yaxis.set_major_formatter(fmt_int); ax.set_xlim(7, 420)
ax.set_xlabel("grid size N (N x N)"); ax.set_ylabel("sweeps to reach eps = 1e-4")
ax.set_title("Sweeps needed grow quickly with the grid size")
show(fig)

Figure 5. Bytes copied GPU to CPU (sheet Transfers): The workbook computes both byte counts with formulas from the sweep counts.

In [ ]:
tr = book["Transfers"].dropna(subset=["ratio"])
n, ours, naive = tr.n.astype(int), tr["bytes, this project"], tr["bytes, grid every sweep"]
fig, ax = plt.subplots(figsize=(6.4, 2.9))
ax.loglog(n, naive, "o-", color=MUTED, mec=SURFACE, mew=1.2)
ax.loglog(n, ours, "o-", color=BACK["cuda_global"], mec=SURFACE, mew=1.2)
label(ax, n.iloc[3], naive.iloc[3], f"copy the grid every sweep: {naive.iloc[-1] / 1e9:.1f} GB at N = 257", dx=-8, dy=12, ha="right")
label(ax, n.iloc[3], ours.iloc[3], f"our kernels: {ours.iloc[-1] / 1e6:.2f} MB at N = 257", dx=8, dy=-12)
ax.set_xticks(NS); ax.xaxis.set_major_formatter(fmt_int)
ax.set_xlabel("grid size N (case 3c, eps = 1e-4)"); ax.set_ylabel("bytes copied GPU -> CPU per run")
ax.set_title("Keeping the grid on the GPU removes almost all copying")
show(fig)
print(f"Ratio at N = 257: {tr.ratio.iloc[-1]:,.0f}x less data")

Figure 6. Time per sweep on the Tesla T4 (sheet T4_benchmark): The CPU time grows with the cell count; both GPU kernels stay far lower.

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 3.0))
for b, dy in zip(BACK, (0, -8, 8)):
    n, ms = t4(b)
    ax.loglog(n, ms, "o-", color=BACK[b], mec=SURFACE, mew=1.2)
    label(ax, n[-1], ms[-1], f"{BACK_NAME[b]}: {ms[-1]:.2f} ms", dy=dy)
ax.set_xticks([256, 512, 1024, 2048]); ax.xaxis.set_major_formatter(fmt_int); ax.set_xlim(220, 5200)
ax.set_xlabel("grid size N (N x N)"); ax.set_ylabel("ms per sweep (log)")
ax.set_title("Time per sweep on a Tesla T4 (check every sweep)")
show(fig)

Figure 7. Speed-up over the CPU (sheet T4_benchmark): Solid: check every sweep; dashed: every 50 sweeps. 139x at N = 2048 (239x with check every 50).

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 3.0))
for b in ("cuda_global", "cuda_shared"):
    for chk, ls in ((1, "-"), (50, "--")):
        n, sp = t4(b, chk, "speedup_vs_cpu")
        ax.semilogx(n, sp, "o" + ls, color=BACK[b], mec=SURFACE, mew=1.2)
        label(ax, n[-1], sp[-1], f"{b.split('_')[1]}, check every {chk}: {sp[-1]:.0f}x", dy=6 if b == "cuda_global" else -6)
ax.set_xticks([256, 512, 1024, 2048]); ax.xaxis.set_major_formatter(fmt_int); ax.set_xlim(220, 9000)
ax.set_xlabel("grid size N (N x N)"); ax.set_ylabel("speed-up over the CPU (x)")
ax.set_title("The bigger the grid, the bigger the GPU advantage")
show(fig)

Figure 8. Shared-memory vs global-memory kernel (sheet Shared_vs_global): Shared is slightly faster only at N = 256 and 512; 12-19% slower at N = 1024 and 2048.

In [ ]:
sg = book["Shared_vs_global"]
ns = [256, 512, 1024, 2048]; w = 0.36
fig, ax = plt.subplots(figsize=(6.4, 2.8))
for k, (chk, col) in enumerate(((1, BACK["cuda_shared"]), (50, BACK_LIGHT["cuda_shared"]))):
    ratio = sg[sg.check_every == chk].sort_values("n")["global / shared"].to_numpy()
    xs = np.arange(len(ns)) + (k - 0.5) * w
    ax.bar(xs, ratio, width=w, color=col, edgecolor=SURFACE, linewidth=1.5, label=f"check every {chk}")
    for x, r in zip(xs, ratio):
        ax.text(x, r + 0.015, f"{r:.2f}", ha="center", fontsize=7.5, color=INK)
ax.axhline(1.0, color=INK2, lw=1); ax.text(3.62, 1.01, "equal speed", fontsize=8, color=INK2, ha="right", va="bottom")
ax.set_xticks(range(len(ns))); ax.set_xticklabels([f"N = {n}" for n in ns]); ax.set_ylim(0.6, 1.2)
ax.set_ylabel("global time / shared time"); ax.legend(loc="upper right", ncol=2)
ax.set_title("Shared-memory tiling vs global memory (above 1 = shared is faster)")
show(fig)

Figure 9. Effective memory bandwidth (sheet T4_benchmark): At N = 2048 the GPU reaches about 230 GB/s of the T4's 320 GB/s peak; the CPU stays near 1-2 GB/s.

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 3.0))
for b in ("cuda_global", "cuda_shared"):
    for chk, ls in ((1, "-"), (50, "--")):
        n, gb = t4(b, chk, "effective_gb_s")
        ax.semilogx(n, gb, "o" + ls, color=BACK[b], mec=SURFACE, mew=1.2)
        label(ax, n[-1], gb[-1], f"{b.split('_')[1]}, check {chk}: {gb[-1]:.0f} GB/s", dy=7 if b == "cuda_global" else -7)
n, gb = t4("cpu_reference", 1, "effective_gb_s")
ax.semilogx(n, gb, "o-", color=BACK["cpu_reference"], mec=SURFACE, mew=1.2); label(ax, n[-1], gb[-1], f"CPU: {gb[-1]:.1f} GB/s")
ax.set_xticks([256, 512, 1024, 2048]); ax.xaxis.set_major_formatter(fmt_int); ax.set_xlim(220, 9000)
ax.set_xlabel("grid size N (N x N)"); ax.set_ylabel("effective bandwidth (GB/s)")
ax.set_title("Bigger grids keep the GPU's memory busier")
show(fig)

Figure 10. Tuning at N = 1024 (sheet T4_tuning): float32 is about 1.7x faster; block shape and cells per thread change the time by under 8%.

In [ ]:
combos = [(b, c) for b in ("16x16", "32x8", "32x16") for c in (1, 2, 4)]
fig, axes = plt.subplots(1, 2, figsize=(7.4, 3.0), sharey=True, constrained_layout=True)
for ax, be in zip(axes, ("cuda_global", "cuda_shared")):
    for k, (dt, col) in enumerate((("float64", BACK[be]), ("float32", BACK_LIGHT[be]))):
        sel = tune[(tune.backend == be) & (tune.dtype == dt) & (tune.check_every == 50)]
        vals = [sel[(sel.block == b) & (sel.coarsen == c)].ms_per_sweep.iloc[0] for b, c in combos]
        xs = np.arange(len(combos)) + (k - 0.5) * 0.4
        ax.bar(xs, vals, width=0.4, color=col, edgecolor=SURFACE, linewidth=1.2, label=dt)
        if dt == "float32":
            i = int(np.argmin(vals))
            ax.annotate(f"best\n{vals[i]:.4f}", (xs[i], vals[i]), xytext=(0, 3), textcoords="offset points", ha="center", fontsize=7)
    ax.set_xticks(range(len(combos))); ax.set_xticklabels([f"{b}\n{c} cell" for b, c in combos], fontsize=6.5)
    ax.set_title("global memory" if be == "cuda_global" else "shared memory", fontsize=9); ax.legend(loc="upper left", ncol=2)
axes[0].set_ylabel("ms per sweep (N = 1024, check 50)")
show(fig)

Figure 11. Centre temperature sweep by sweep (sheet Centre_trajectory): Fast at first, then ever more slowly towards 75, 40 and 25 (dashed).

In [ ]:
ct = book["Centre_trajectory"]
fig, ax = plt.subplots(figsize=(6.4, 2.9))
for case in CASE:
    s = ct[case].dropna()
    ax.plot(ct.sweep[: len(s)], s, color=CASE[case])
    ax.axhline(EXACT[case], color=CASE[case], lw=1, ls="--", alpha=0.7)
    label(ax, len(s), EXACT[case], f"{case} -> {EXACT[case]:g}")
ax.set_xlim(0, len(ct) * 1.32); ax.xaxis.set_major_formatter(fmt_int)
ax.set_xlabel("sweep (65 x 65 grid)"); ax.set_ylabel("centre temperature")
ax.set_title("The centre cell creeps up to its exact value")
show(fig)

Figure 12. Sweeps against tolerance (sheet Sweeps_vs_eps): On the 65 x 65 grid every 10x smaller eps costs about 1,910 more sweeps.

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 2.9))
for col, (n, r) in zip(RAMP4[1:], eps.groupby("n")):
    ax.semilogx(r.eps, r.sweeps, "o-", color=col, mec=SURFACE, mew=1.2)
    label(ax, r.eps.iloc[-1], r.sweeps.iloc[-1], f"N = {n}", dx=-6, dy=8, ha="right")
ax.invert_xaxis(); ax.yaxis.set_major_formatter(fmt_int)
ax.set_xlabel("tolerance eps (smaller to the right)"); ax.set_ylabel("sweeps (case 3c)")
ax.set_title("Each extra digit of accuracy costs the same number of sweeps")
show(fig)

Figure 13. True error at the centre (sheets Sweeps_vs_eps and Sweeps_vs_N): The true error is far larger than eps and grows 4x per doubling of N, so larger grids need a smaller eps.

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(7.2, 2.9), constrained_layout=True)
for col, (n, r) in zip(RAMP4[1:], eps.groupby("n")):
    a1.loglog(r.eps, r["centre error"], "o-", color=col, mec=SURFACE, mew=1.2)
    label(a1, r.eps.iloc[2], r["centre error"].iloc[2], f"N = {n}", dy=7)
a1.loglog([1e-8, 1e-1], [1e-8, 1e-1], color=MUTED, lw=1, ls="--"); a1.text(1.2e-3, 3e-4, "error = eps", color=INK2, fontsize=8, rotation=30)
a1.invert_xaxis(); a1.set_xlabel("tolerance eps"); a1.set_ylabel("centre error |T - 25|"); a1.set_title("Error at the centre vs eps", fontsize=9)
r = svn[svn.case == "3c hot edge"].sort_values("n")
a2.loglog(r.n, r["centre error"], "o-", color=CASE["3c hot edge"], mec=SURFACE, mew=1.2)
for x, y in zip(r.n, r["centre error"]):
    a2.annotate(f"{y:.3f}" if y < 0.1 else f"{y:.2f}", (x, y), xytext=(0, 7), textcoords="offset points", ha="center", fontsize=7, color=INK2)
a2.set_xticks(NS); a2.xaxis.set_major_formatter(fmt_int); a2.set_ylim(top=4)
a2.set_xlabel("grid size N"); a2.set_ylabel("centre error at eps = 1e-4"); a2.set_title("Same eps, bigger grid: larger error", fontsize=9)
show(fig)